<a href="https://colab.research.google.com/github/Enzopeter/GS/blob/main/GS_Sergio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Análise do Perfil Socioeconômico de Usuários dos Serviços Públicos de Saúde e Educação


Primeira parte: Neste primeiro notebook, não treinaremos um modelo. Vamos confirmar a estrutura, códigos e valores ausentes, para não interpretar incorretamente as respostas. Os resultados são históricos (2008).

Abrindo os arquivos

In [ ]:
import requests
from pathlib import Path

usuario = "SEU_USUARIO"
repositorio = "GS-Modelagem-Linear"

base_url = (
    f"https://raw.githubusercontent.com/"
    f"{usuario}/{repositorio}/main/dados"
)

arquivos = [
    "Dados.zip",
    "Dicionarios.zip",
    "Questionario.zip"
]

for nome in arquivos:
    url = f"{base_url}/{nome}"

    resposta = requests.get(url, timeout=180)
    resposta.raise_for_status()

    Path(nome).write_bytes(resposta.content)

    print(f"{nome} baixado com sucesso!")

Arquivos recebidos: []


Analisando os registros de pessoas

In [ ]:
from pathlib import Path
from zipfile import ZipFile

for nome in ["Dados.zip", "Dicionarios.zip", "Questionario.zip"]:
    caminho = Path(nome)
    if not caminho.exists():
        print(f"Arquivo ainda não encontrado: {nome}")
        continue
    print(f"\n{nome} ({caminho.stat().st_size / 1_000_000:.1f} MB)")
    with ZipFile(caminho) as zip_file:
        for item in zip_file.infolist():
            print(f"  {item.filename}: {item.file_size / 1_000_000:.1f} MB")

Arquivo ainda não encontrado: Dados.zip
Arquivo ainda não encontrado: Dicionarios.zip
Arquivo ainda não encontrado: Questionario.zip


Lendo as posições das variáveis

In [ ]:
import re
import pandas as pd

with ZipFile("Dicionarios.zip") as zip_file:
    layout = zip_file.read("input PES2008.txt").decode("latin1")

# Posição inicial (a partir de 1), nome da variável e tamanho do campo
padrao = re.compile(r"@\s*(\d+)\s+([A-Za-z][A-Za-z0-9_]*)\s+(\$?\d+)\.")
posicoes = {}
for posicao, nome, formato in padrao.findall(layout):
    inicio = int(posicao) - 1    # Python começa a contar em zero
    tamanho = int(formato.replace("$", ""))
    posicoes[nome.upper()] = (inicio, inicio + tamanho)

variaveis = ["UF", "V0302", "V8005", "V0602", "V6002", "V4742", "V4745", "V1350", "V1366"]
print("Variáveis localizadas:")
for nome in variaveis:
    print(nome, posicoes.get(nome, "NÃO ENCONTRADA"))

FileNotFoundError: [Errno 2] No such file or directory: 'Dicionarios.zip'

Carregando apenas as colunas que serão relevantes

In [ ]:
nomes = [v for v in variaveis if v in posicoes]
registros = []
with ZipFile("Dados.zip") as zip_file:
    with zip_file.open("PES2008.TXT") as arquivo:
        for linha in arquivo:
            registros.append({
                nome: linha[inicio:fim].decode("latin1").strip() or None
                for nome in nomes
                for inicio, fim in [posicoes[nome]]
            })

df = pd.DataFrame(registros)
print(f"Registros carregados: {len(df):,}")
print(f"Variáveis carregadas: {len(df.columns)}")
display(df.head())

Verificação de respostas disponíveis e campos vazios

In [ ]:
descricao = {
    "UF": "Unidade da Federação",
    "V0302": "Sexo",
    "V8005": "Idade",
    "V0602": "Frequência escolar",
    "V6002": "Rede de ensino",
    "V4742": "Renda domiciliar per capita",
    "V4745": "Nível de instrução",
    "V1350": "Procurou atendimento de saúde nas últimas duas semanas",
    "V1366": "Atendimento feito pelo SUS"
}
for coluna in df.columns:
    print(f"\n{coluna} — {descricao.get(coluna, '')}")
    print("Ausentes:", df[coluna].isna().sum())
    display(df[coluna].value_counts(dropna=False).head(12).rename_axis("Código original").reset_index(name="Quantidade"))

Consulta do dicionário original

In [ ]:
%pip -q install xlrd

In [ ]:
from io import BytesIO

with ZipFile("Dicionarios.zip") as zip_file:
    nomes_xls = [nome for nome in zip_file.namelist()
                 if nome.lower().endswith(".xls") and "pessoas" in nome.lower()]
    print("Dicionário encontrado:", nomes_xls)
    conteudo_xls = zip_file.read(nomes_xls[0])

excel = pd.ExcelFile(BytesIO(conteudo_xls), engine="xlrd")
print("Abas:", excel.sheet_names)

busca = ["V0302", "V8005", "V0602", "V6002", "V4742", "V4745", "V1350", "V1366"]
for aba in excel.sheet_names:
    tabela = pd.read_excel(excel, sheet_name=aba, header=None, dtype=str)
    encontrou = tabela.astype(str).apply(
        lambda coluna: coluna.str.contains("|".join(busca), case=False, regex=True, na=False)
    ).any(axis=1)
    indices = tabela.index[encontrou].tolist()
    if indices:
        print(f"\nABA: {aba} | Ocorrências: {len(indices)}")
        for i in indices[:25]:
            print(f"Linha {i + 1}")
            display(tabela.iloc[max(0, i-1):min(len(tabela), i+8)].fillna(""))

Nesta etapa, utilizaremos dados reais da PNAD 2008. Os códigos de rede escolar e atendimento SUS foram conferidos no questionário original.

Perguntas para estudo:

1-Entre pessoas de 5 a 24 anos que frequentam escola ou creche, quais características estão associadas à frequência na rede pública em comparação à particular?

2-Entre pessoas que receberam um atendimento de saúde registrado nas últimas duas semanas, quais características estão associadas ao atendimento via SUS em comparação ao não SUS?

Preparação das colunas necessárias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

colunas_necessarias = ['UF','V0302','V8005','V0602','V6002','V4742','V4745','V1350','V1366']
faltando = [c for c in colunas_necessarias if c not in df.columns]
if faltando:
    raise ValueError(f'Colunas ausentes: {faltando}. Execute primeiro as células da Etapa 1.')

dados = df[colunas_necessarias].copy()
dados['idade'] = pd.to_numeric(dados['V8005'], errors='coerce')
dados['renda_per_capita'] = pd.to_numeric(dados['V4742'], errors='coerce')
dados.loc[dados['V4742'] == '999999999999', 'renda_per_capita'] = float('nan')
dados['sexo'] = dados['V0302'].map({'2':'Masculino','4':'Feminino'})
dados['frequenta_escola'] = dados['V0602'].map({'2':'Sim','4':'Não'})
print('Registros originais:',len(dados))
print('Idades sem informação:',dados['idade'].isna().sum())
print('Renda sem informação:',dados['renda_per_capita'].isna().sum())
display(dados[['idade','sexo','renda_per_capita','frequenta_escola']].head())


Criando bases de análise

In [ ]:
educacao = dados.loc[
    dados['idade'].between(5, 24) & dados['V6002'].isin(['2', '4'])
].copy()
educacao['rede_ensino'] = educacao['V6002'].map({'2': 'Pública', '4': 'Particular'})
educacao['classe'] = (educacao['V6002'] == '2').astype(int)

saude = dados.loc[dados['V1366'].isin(['2', '4'])].copy()
saude['tipo_atendimento'] = saude['V1366'].map({'2': 'SUS', '4': 'Não SUS'})
saude['classe'] = (saude['V1366'] == '2').astype(int)

print('Pessoas selecionadas para educação:', len(educacao))
print('Pessoas selecionadas para saúde:', len(saude))
print('\nClasses de educação:')
display(educacao['rede_ensino'].value_counts().rename_axis('Rede').reset_index(name='Quantidade'))
print('\nClasses de saúde:')
display(saude['tipo_atendimento'].value_counts().rename_axis('Atendimento').reset_index(name='Quantidade'))


Gráficos comparando as classes

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
educacao['rede_ensino'].value_counts().reindex(['Pública','Particular']).plot(kind='bar', ax=ax)
ax.set_title('Rede de ensino — pessoas de 5 a 24 anos matriculadas')
ax.set_ylabel('Registros na amostra')
ax.set_xlabel('Rede de ensino')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
saude['tipo_atendimento'].value_counts().reindex(['SUS','Não SUS']).plot(kind='bar', ax=ax)
ax.set_title('Tipo de atendimento — respostas válidas sobre SUS')
ax.set_ylabel('Registros na amostra')
ax.set_xlabel('Atendimento')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


Comparando idade, sexo e renda

In [ ]:
def resumo_grupo(tabela, coluna_grupo):
    return (tabela.groupby(coluna_grupo)
            .agg(registros=('classe','size'),
                 idade_media=('idade','mean'),
                 renda_mediana=('renda_per_capita','median'),
                 renda_ausente=('renda_per_capita', lambda x: x.isna().sum()))
            .round(2))

print('EDUCAÇÃO')
display(resumo_grupo(educacao, 'rede_ensino'))
print('SAÚDE')
display(resumo_grupo(saude, 'tipo_atendimento'))

print('Sexo x rede de ensino (percentual em cada grupo de rede):')
display((pd.crosstab(educacao['rede_ensino'], educacao['sexo'], normalize='index')*100).round(1))
print('Sexo x atendimento (percentual em cada grupo de atendimento):')
display((pd.crosstab(saude['tipo_atendimento'], saude['sexo'], normalize='index')*100).round(1))


Verificação de quantidade e valores faltantes

In [ ]:
variaveis_modelo = ['idade','sexo','UF','V4745','renda_per_capita']
for nome, tabela in [('Educação', educacao), ('Saúde', saude)]:
    print(f'\n{nome}: {len(tabela)} registros')
    print('Distribuição da classe:')
    display(tabela['classe'].value_counts(normalize=True).mul(100).round(2).rename('Percentual (%)'))
    print('Valores ausentes:')
    display(tabela[variaveis_modelo].isna().sum().rename('Ausentes'))

print('Etapa 2 concluída. Guarde as tabelas educacao e saude para a próxima etapa.')


Modelos de classificação (Machine Learning)

Vamos treinar dois modelos de regressão logística, um para cada público: (1) alunos de 5 a 24 anos que frequentavam rede pública (1) ou particular (0); (2) pessoas com resposta de atendimento SUS (1) ou não SUS (0).

Preparar bibliotecas e variáveis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

colunas_numericas = ['idade', 'renda_per_capita']
colunas_categoricas = ['sexo', 'UF', 'V4745']
colunas_entrada = colunas_numericas + colunas_categoricas

for nome, tabela in [('Educação', educacao), ('Saúde', saude)]:
    colunas_faltantes = set(colunas_entrada + ['classe']) - set(tabela.columns)
    if colunas_faltantes:
        raise ValueError(f'{nome}: faltam as colunas {colunas_faltantes}. Execute a Etapa 2.')
    print(f'{nome}: {len(tabela):,} registros | {tabela["classe"].mean():.1%} de classe 1')


Criar e treinar um modelo para cada serviço

In [ ]:
def treinar_modelo(tabela, nome):
    X = tabela[colunas_entrada].copy()
    y = tabela['classe'].astype(int).copy()

    # Previne que números venham como texto.
    for coluna in colunas_numericas:
        X[coluna] = pd.to_numeric(X[coluna], errors='coerce')

    # Valores categóricos ficam como texto; ausências são imputadas no Pipeline.
    for coluna in colunas_categoricas:
        X[coluna] = X[coluna].astype('string').fillna(pd.NA).astype(object)
        X[coluna] = X[coluna].where(pd.notna(X[coluna]), np.nan)

    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    processamento_numerico = Pipeline([
        ('preencher', SimpleImputer(strategy='median')),
        ('padronizar', StandardScaler())
    ])
    processamento_categorico = Pipeline([
        ('preencher', SimpleImputer(strategy='most_frequent')),
        ('codificar', OneHotEncoder(handle_unknown='ignore'))
    ])
    preparacao = ColumnTransformer([
        ('numericas', processamento_numerico, colunas_numericas),
        ('categoricas', processamento_categorico, colunas_categoricas)
    ])
    modelo = Pipeline([
        ('preparacao', preparacao),
        ('classificador', LogisticRegression(max_iter=1000, random_state=42))
    ])
    modelo.fit(X_treino, y_treino)
    previsoes = modelo.predict(X_teste)
    classe_majoritaria = int(y_treino.mode().iloc[0])
    acuracia_baseline = (y_teste == classe_majoritaria).mean()

    resultado = {
        'Serviço': nome,
        'Treino': len(y_treino),
        'Teste': len(y_teste),
        'Acurácia': accuracy_score(y_teste, previsoes),
        'Precisão': precision_score(y_teste, previsoes, zero_division=0),
        'Revocação': recall_score(y_teste, previsoes, zero_division=0),
        'F1-score': f1_score(y_teste, previsoes, zero_division=0),
        'Baseline (acurácia)': acuracia_baseline
    }
    return modelo, y_teste, previsoes, resultado

modelo_educacao, y_teste_educacao, pred_educacao, res_educacao = treinar_modelo(educacao, 'Educação')
modelo_saude, y_teste_saude, pred_saude, res_saude = treinar_modelo(saude, 'Saúde')

resultados = pd.DataFrame([res_educacao, res_saude])
display(resultados.style.format({
    'Acurácia': '{:.2%}', 'Precisão': '{:.2%}', 'Revocação': '{:.2%}',
    'F1-score': '{:.2%}', 'Baseline (acurácia)': '{:.2%}'
}))


Interpretar acurácia, precisão e revocação

In [ ]:
print('EDUCAÇÃO — 0: Particular | 1: Pública')
print(classification_report(y_teste_educacao, pred_educacao,
                            labels=[0, 1], target_names=['Particular', 'Pública'], zero_division=0))
print('SAÚDE — 0: Não SUS | 1: SUS')
print(classification_report(y_teste_saude, pred_saude,
                            labels=[0, 1], target_names=['Não SUS', 'SUS'], zero_division=0))


Matrizes de confusão

In [ ]:
for nome, y_real, y_pred, rotulos in [
    ('Educação', y_teste_educacao, pred_educacao, ['Particular', 'Pública']),
    ('Saúde', y_teste_saude, pred_saude, ['Não SUS', 'SUS'])
]:
    matriz = confusion_matrix(y_real, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=rotulos).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d'
    )
    ax.set_title(f'Matriz de confusão — {nome}')
    plt.tight_layout()
    plt.show()

Comparação com o baseline e conclusões provisórias

In [ ]:
for _, r in resultados.iterrows():
    ganho_pp = 100 * (r['Acurácia'] - r['Baseline (acurácia)'])
    print(f"{r['Serviço']}: acurácia={r['Acurácia']:.1%}; baseline={r['Baseline (acurácia)']:.1%}; "
          f"diferença={ganho_pp:+.1f} ponto(s) percentual(is)")